# 05 — Difficulty Rating with Teacher Model (Few-shot)

Re-rate questions generated by the 7B model using **few-shot prompting** at `temperature=0.0`
for deterministic, consistent `difficulty_teacher` labels (1=easy / 2=medium / 3=hard).

**Run on:** A100 server (after SSH tunnel for port 8001 is active)  
**Model:** `Qwen/Qwen2.5-7B-Instruct-AWQ` on port 8001  
*(14B/32B require 2g.20gb+ which is not available on this account)*

## Quick Start
1. Start teacher: `sbatch ~/ku_prep_arena/ai/scripts/slurm_teacher.sh`
2. Open SSH tunnel (local): `ssh -N -L 8001:dgx-02:8001 aip04@br2.paas.ku.ac.th`
3. Run this notebook cell by cell

**Input:** `dataset/generated/*.json`  
**Output:** `dataset/rated/*.json` (same structure + `difficulty_teacher` + `difficulty_reason` fields)

In [18]:
import os, json, time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from openai import OpenAI
from tqdm.notebook import tqdm

# ── Connect to Teacher vLLM (port 8001) ─────────────────────────────────────
BASE_URL = os.getenv("AI_BASE_URL", "http://localhost:8001/v1")
API_KEY  = os.getenv("AI_API_KEY",  "none")
MODEL    = os.getenv("AI_MODEL",    "Qwen/Qwen2.5-7B-Instruct-AWQ")

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

# Quick health check
try:
    models = client.models.list()
    print(f"✅ Connected to: {BASE_URL}")
    print(f"✅ Available model: {models.data[0].id if models.data else 'none'}")
except Exception as e:
    print(f"❌ Cannot connect: {e}")
    print("   → ตรวจสอบว่า slurm_teacher.sh รันอยู่และ SSH tunnel port 8001 เปิดแล้ว")
    raise

✅ Connected to: http://localhost:8001/v1
✅ Available model: Qwen/Qwen2.5-7B-Instruct-AWQ


## Config

In [19]:
# ── Paths ───────────────────────────────────────────────────────────────────
GEN_DIR  = Path("../dataset/generated")   # input: 7B-generated questions
RATED_DIR = Path("../dataset/rated")       # output: teacher-rated questions

RATED_DIR.mkdir(parents=True, exist_ok=True)

gen_files = sorted(GEN_DIR.glob("*.json"))
print(f"Generated files : {len(gen_files)}")
print(f"Output dir      : {RATED_DIR}")
for f in gen_files:
    print(f"  {f.name}")

Generated files : 11
Output dir      : ../dataset/rated
  670E74B8BEC523762D0AC71DAD2B48B65BA6A497_232m0077 fund chem midterm_compressed_bricks.json
  670E74B8BEC523762D0AC71DAD2B48B65BA6A497_232m0077 fund chem midterm_compressed_flappy.json
  670E74B8BEC523762D0AC71DAD2B48B65BA6A497_232m0077 fund chem midterm_compressed_racer.json
  670E74B8BEC523762D0AC71DAD2B48B65BA6A497_232m0077 fund chem midterm_compressed_shooter.json
  670E74B8BEC523762D0AC71DAD2B48B65BA6A497_232m0077 fund chem midterm_compressed_snake.json
  Calculus_1_Midterm_สำเนา_compressed_shooter.json
  python101_workbook_v1.0.2_compressed_bricks.json
  python101_workbook_v1.0.2_compressed_flappy.json
  python101_workbook_v1.0.2_compressed_racer.json
  python101_workbook_v1.0.2_compressed_shooter.json
  python101_workbook_v1.0.2_compressed_snake.json


## Few-shot prompt setup

In [20]:
RATE_SYSTEM = """คุณเป็น expert ประเมินความยากของข้อสอบแบบปรนัย ตอบด้วย JSON เท่านั้น ห้ามมีข้อความอื่น

ระดับความยาก:
1 = easy   : จำตรงๆ หรือนิยามเดียว ไม่ต้องวิเคราะห์
2 = medium : ต้องเข้าใจ เปรียบเทียบ หรือประยุกต์ใช้แนวคิด
3 = hard   : ต้องวิเคราะห์ สังเคราะห์ หรือประเมินในสถานการณ์ซับซ้อน

ตอบรูปแบบ: {\"difficulty\": 1|2|3, \"reason\": \"อธิบายสั้นๆ ว่าทำไม\"}"""

FEW_SHOT_EXAMPLES = [
    {
        "question": "กฎข้อที่ 2 ของนิวตันคืออะไร?",
        "choices": ["F = ma", "F = mv", "E = mc²", "p = mv"],
        "answer": '{"difficulty": 1, "reason": "ถามตรงๆ ว่าสูตรคืออะไร จำได้ก็ตอบได้ทันที"}'
    },
    {
        "question": "Which data structure uses LIFO ordering?",
        "choices": ["Stack", "Queue", "Linked List", "Heap"],
        "answer": '{"difficulty": 1, "reason": "Direct recall of Stack LIFO property — no analysis needed"}'
    },
    {
        "question": "อัลกอริทึมใดที่รักษาลำดับของ elements ที่มีค่าเท่ากัน (stable sort)?",
        "choices": ["Merge Sort", "Quick Sort", "Heap Sort", "Shell Sort"],
        "answer": '{"difficulty": 2, "reason": "ต้องเข้าใจ property ของ stable sort และเปรียบเทียบอัลกอริทึม ไม่ใช่แค่จำชื่อ"}'
    },
    {
        "question": "Which OSI layer is responsible for end-to-end error recovery?",
        "choices": ["Transport", "Network", "Data Link", "Session"],
        "answer": '{"difficulty": 2, "reason": "Requires understanding of OSI layer responsibilities, not just memorizing layer names"}'
    },
    {
        "question": "เมื่อ array เกือบ sorted แล้ว อัลกอริทึมใดมี Big-O ดีที่สุดในทางปฏิบัติ?",
        "choices": ["Insertion Sort O(n)", "Merge Sort O(n log n)", "Quick Sort O(n²)", "Heap Sort O(n log n)"],
        "answer": '{"difficulty": 3, "reason": "ต้องวิเคราะห์ best-case behaviour ของแต่ละ algorithm ในสถานการณ์เฉพาะ (nearly-sorted)"}'
    },
    {
        "question": "A system needs 99.99% uptime. Which combination of strategies best achieves this?",
        "choices": [
            "Active-active clustering + geographic replication",
            "Single primary with hot standby",
            "Frequent backups + fast restore procedure",
            "Load balancer with health checks only"
        ],
        "answer": '{"difficulty": 3, "reason": "Requires synthesizing multiple availability concepts and evaluating trade-offs for a specific SLA target"}'
    },
]

ABCD = ['A', 'B', 'C', 'D']

def build_few_shot_user(q: dict) -> str:
    lines = []
    for ex in FEW_SHOT_EXAMPLES:
        lines.append(f"Q: {ex['question']}")
        for i, c in enumerate(ex["choices"][:4]):   # cap at 4
            lines.append(f"  {ABCD[i]}. {c}")
        lines.append(f"A: {ex['answer']}")
        lines.append("")
    lines.append(f"Q: {q['question']}")
    for i, c in enumerate(q.get("choices", [])[:4]):  # cap at 4
        lines.append(f"  {ABCD[i]}. {c}")
    lines.append("A:")
    return "\n".join(lines)

print("✅ Few-shot prompt ready")

✅ Few-shot prompt ready


## `rate_question` function

In [21]:
import re

def rate_question(q: dict, retries: int = 3) -> dict:
    """
    Send a question to the 30B teacher and return a dict with:
      difficulty_teacher: int (1/2/3)
      difficulty_reason: str
    Falls back to the original 'difficulty' field on failure.
    """
    messages = [
        {"role": "system", "content": RATE_SYSTEM},
        {"role": "user",   "content": build_few_shot_user(q)},
    ]
    for attempt in range(retries):
        try:
            res = client.chat.completions.create(
                model=MODEL,
                messages=messages,
                temperature=0.0,   # deterministic for rating
                max_tokens=120,
            )
            raw = res.choices[0].message.content or "{}"
            # Strip markdown fences if any
            raw = re.sub(r"^```[a-z]*\n?", "", raw, flags=re.IGNORECASE)
            raw = re.sub(r"\n?```$", "", raw, flags=re.IGNORECASE).strip()
            parsed = json.loads(raw)
            diff = int(parsed.get("difficulty", q.get("difficulty", 2)))
            diff = max(1, min(3, diff))  # clamp to [1,3]
            return {
                "difficulty_teacher": diff,
                "difficulty_reason": parsed.get("reason", ""),
            }
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(2 ** attempt)   # exponential backoff
            else:
                return {
                    "difficulty_teacher": q.get("difficulty", 2),
                    "difficulty_reason": f"fallback (error: {e})",
                }

# Quick smoke test
test_q = {
    "question": "What is 2 + 2?",
    "choices": ["3", "4", "5", "6"],
}
result = rate_question(test_q)
print(f"Smoke test → {result}")

Smoke test → {'difficulty_teacher': 1, 'difficulty_reason': 'จำตรงๆ ไม่ต้องวิเคราะห์'}


## Batch rating — all generated files

In [22]:
MAX_WORKERS = 4   # concurrent requests to 30B teacher

def rate_file(gen_path: Path) -> dict:
    """Rate all questions in one generated JSON file and save to rated/."""
    out_path = RATED_DIR / gen_path.name
    if out_path.exists():
        return {"file": gen_path.name, "status": "skipped (already rated)"}

    questions = json.loads(gen_path.read_text(encoding="utf-8"))
    rated = []
    for q in questions:
        rating = rate_question(q)
        rated.append({**q, **rating})

    out_path.write_text(json.dumps(rated, ensure_ascii=False, indent=2), encoding="utf-8")
    return {"file": gen_path.name, "status": "ok", "n": len(rated)}

# ── Parallel batch ───────────────────────────────────────────────────────────
gen_files = sorted(GEN_DIR.glob("*.json"))
print(f"Rating {len(gen_files)} files with {MAX_WORKERS} workers...")

results = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    futures = {pool.submit(rate_file, f): f.name for f in gen_files}
    for fut in tqdm(as_completed(futures), total=len(futures), desc="Files"):
        res = fut.result()
        results.append(res)
        print(f"  {res['file']} → {res['status']}")

ok = [r for r in results if r.get("status") == "ok"]
print(f"\nDone: {len(ok)}/{len(results)} files rated successfully")

Rating 11 files with 4 workers...


Files:   0%|          | 0/11 [00:00<?, ?it/s]

  python101_workbook_v1.0.2_compressed_snake.json → skipped (already rated)
  python101_workbook_v1.0.2_compressed_shooter.json → skipped (already rated)
  python101_workbook_v1.0.2_compressed_racer.json → skipped (already rated)
  python101_workbook_v1.0.2_compressed_bricks.json → skipped (already rated)
  670E74B8BEC523762D0AC71DAD2B48B65BA6A497_232m0077 fund chem midterm_compressed_bricks.json → skipped (already rated)
  python101_workbook_v1.0.2_compressed_flappy.json → skipped (already rated)
  670E74B8BEC523762D0AC71DAD2B48B65BA6A497_232m0077 fund chem midterm_compressed_shooter.json → skipped (already rated)
  670E74B8BEC523762D0AC71DAD2B48B65BA6A497_232m0077 fund chem midterm_compressed_snake.json → skipped (already rated)
  670E74B8BEC523762D0AC71DAD2B48B65BA6A497_232m0077 fund chem midterm_compressed_flappy.json → skipped (already rated)
  670E74B8BEC523762D0AC71DAD2B48B65BA6A497_232m0077 fund chem midterm_compressed_racer.json → skipped (already rated)
  Calculus_1_Midterm_ส

## Stats — compare original vs teacher-rated distribution

In [23]:
from collections import Counter

orig_dist: Counter = Counter()
teacher_dist: Counter = Counter()
total_q = 0

for rated_path in sorted(RATED_DIR.glob("*.json")):
    questions = json.loads(rated_path.read_text(encoding="utf-8"))
    for q in questions:
        total_q += 1
        orig_dist[q.get("difficulty", "N/A")] += 1
        teacher_dist[q.get("difficulty_teacher", "N/A")] += 1

print(f"Total questions rated: {total_q}")
print()
print("Level  | Original (self) | Teacher (few-shot)")
print("-------|-----------------|-------------------")
for level in [1, 2, 3, "N/A"]:
    o = orig_dist.get(level, 0)
    t = teacher_dist.get(level, 0)
    pct_o = f"{o/total_q*100:.1f}%" if total_q else "-"
    pct_t = f"{t/total_q*100:.1f}%" if total_q else "-"
    label = {1: "easy  ", 2: "medium", 3: "hard  "}.get(level, "N/A   ")
    print(f"{label} |  {o:4d} {pct_o:>6}          |  {t:4d} {pct_t:>6}")

# Agreement rate
agree = 0
disagree_examples = []
for rated_path in sorted(RATED_DIR.glob("*.json")):
    questions = json.loads(rated_path.read_text(encoding="utf-8"))
    for q in questions:
        orig = q.get("difficulty")
        teacher = q.get("difficulty_teacher")
        if orig is not None and teacher is not None:
            if orig == teacher:
                agree += 1
            elif len(disagree_examples) < 3:
                disagree_examples.append(q)

print(f"\nAgreement (original vs few-shot): {agree}/{total_q} = {agree/total_q*100:.1f}%" if total_q else "")

if disagree_examples:
    print("\nSample disagreements (few-shot overrode original label):")
    for q in disagree_examples:
        print(f"  Q: {q['question'][:80]}")
        print(f"     original={q.get('difficulty')}  few-shot={q.get('difficulty_teacher')}  reason: {q.get('difficulty_reason','')[:80]}")

Task was destroyed but it is pending!
task: <Task pending name='Task-258' coro=<_async_in_context.<locals>.run_in_context() done, defined at /Users/frank/Vscode/KU_Prep_Arena/ai/venv/lib/python3.13/site-packages/ipykernel/utils.py:57> wait_for=<Task pending name='Task-259' coro=<Kernel.shell_main() running at /Users/frank/Vscode/KU_Prep_Arena/ai/venv/lib/python3.13/site-packages/ipykernel/kernelbase.py:597> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /Users/frank/Vscode/KU_Prep_Arena/ai/venv/lib/python3.13/site-packages/zmq/eventloop/zmqstream.py:563]>
/Users/frank/Vscode/KU_Prep_Arena/ai/venv/lib/python3.13/site-packages/IPython/core/compilerop.py:86: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  return compile(source, filename, symbol, self.flags | PyCF_ONLY_AST, 1)
Task was destroyed but it is pending!
task: <Task pending name='Task-259' coro=<Kernel.shell_main() running at /Users/frank/Vscode/KU_Prep_Arena/ai/venv/lib/python3.13

Total questions rated: 110

Level  | Original (self) | Teacher (few-shot)
-------|-----------------|-------------------
easy   |    31  28.2%          |    83  75.5%
medium |    71  64.5%          |    27  24.5%
hard   |     8   7.3%          |     0   0.0%
N/A    |     0   0.0%          |     0   0.0%

Agreement (original vs few-shot): 44/110 = 40.0%

Sample disagreements (few-shot overrode original label):
  Q: Which of the following represents the correct order of electron configuration fo
     original=2  few-shot=1  reason: จำตรงๆ ว่าลำดับของ electron configuration ของ元素15是什么，直接对应即可
  Q: Which of the following elements has the highest ionization energy according to t
     original=2  few-shot=1  reason: Direct recall from the given options — no analysis needed
  Q: What does the term 'sp3' hybridization refer to?
     original=2  few-shot=1  reason: จำตรงๆ ว่า sp3 หมายถึงอะไร ไม่ต้องวิเคราะห์
